# VeReMi Graph Feature Engineering — FINAL Self-Contained Notebook

This notebook contains the corrected feature-engineering pipeline in **one file**.

Key corrections included:
- no external `src.feature_engine` import;
- active but isolated vehicles are retained as graph nodes;
- zero-edge snapshots are valid;
- initialization-only `snapshot_id = -1` is excluded;
- final graph sequence is snapshots `0..99` = **100 snapshots per archive**;
- `attackerType` remains an edge/message target only;
- receiver state is attached causally at or before `rcvTime`.


In [ ]:
from pathlib import Path
import sys

# Locate repository root when notebook is executed from notebooks/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name in {"notebooks", "analysis", "examples"}:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import duckdb

from src.config import GRAPH_ENGINE_ROOT, GRAPH_FEATURE_ROOT
GRAPH_ROOT = GRAPH_ENGINE_ROOT
FEATURE_ROOT = GRAPH_FEATURE_ROOT
FEATURE_ROOT.mkdir(parents=True, exist_ok=True)
(FEATURE_ROOT / "archives").mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 1.0

archive_root = GRAPH_ROOT / "archives"
archive_dirs = sorted(p for p in archive_root.iterdir() if p.is_dir())

print("Archives found:", len(archive_dirs))
print("Output:", FEATURE_ROOT)

if len(archive_dirs) != 225:
    print("WARNING: expected 225 archives.")


In [ ]:
def _safe_num(s):
    return pd.to_numeric(s, errors="coerce")


def add_snapshot_to_nodes(nodes, edges, window_size):
    out = nodes.copy()
    if out.empty or edges.empty:
        return out
    t0 = float(edges["sendTime"].min())
    out["snapshot_id"] = np.floor(
        (_safe_num(out["time"]) - t0) / window_size
    ).astype("Int64")
    return out


def latest_node_state_per_snapshot(nodes_with_snapshot):
    if nodes_with_snapshot.empty:
        return pd.DataFrame()

    x = nodes_with_snapshot.dropna(
        subset=["snapshot_id", "vehicle_id", "time"]
    ).copy()

    x = x.sort_values(["snapshot_id", "vehicle_id", "time"])

    latest = (
        x.groupby(["snapshot_id", "vehicle_id"], as_index=False, sort=False)
         .tail(1)
         .reset_index(drop=True)
    )

    if {"spd_x", "spd_y"}.issubset(latest.columns):
        latest["speed_magnitude"] = np.sqrt(
            latest["spd_x"]**2 + latest["spd_y"]**2
        )
        latest["motion_angle"] = np.arctan2(
            latest["spd_y"], latest["spd_x"]
        )

    latest = latest.sort_values(
        ["vehicle_id", "snapshot_id"]
    ).reset_index(drop=True)

    g = latest.groupby("vehicle_id", sort=False)

    if "speed_magnitude" in latest.columns:
        latest["delta_speed_magnitude"] = g["speed_magnitude"].diff()

    if {"pos_x", "pos_y"}.issubset(latest.columns):
        dx = g["pos_x"].diff()
        dy = g["pos_y"].diff()
        latest["snapshot_displacement"] = np.sqrt(dx**2 + dy**2)

    if "motion_angle" in latest.columns:
        raw = g["motion_angle"].diff()
        latest["delta_motion_angle"] = np.arctan2(
            np.sin(raw), np.cos(raw)
        )

    return latest


In [ ]:
def add_receiver_state_to_edges(edges, node_states):
    if edges.empty:
        return edges.copy()

    e = edges.copy()
    n = node_states.copy()

    e["rcvTime"] = _safe_num(e["rcvTime"])
    e["receiver"] = _safe_num(e["receiver"]).astype("Int64")

    n["time"] = _safe_num(n["time"])
    n["vehicle_id"] = _safe_num(n["vehicle_id"]).astype("Int64")

    keep = [c for c in [
        "vehicle_id","time","pos_x","pos_y","spd_x","spd_y","speed_magnitude"
    ] if c in n.columns]

    n = n[keep].dropna(subset=["vehicle_id","time"]).copy()

    e = e.sort_values(["rcvTime","receiver"]).reset_index(drop=True)
    n = n.sort_values(["time","vehicle_id"]).reset_index(drop=True)

    merged = pd.merge_asof(
        e, n,
        left_on="rcvTime",
        right_on="time",
        left_by="receiver",
        right_by="vehicle_id",
        direction="backward",
        allow_exact_matches=True
    )

    rename = {}
    for c in ["pos_x","pos_y","spd_x","spd_y","speed_magnitude"]:
        if c in merged.columns:
            rename[c] = f"receiver_{c}"

    merged = merged.rename(columns=rename)
    merged["receiver_state_age"] = merged["rcvTime"] - merged["time"]

    return merged


def engineer_edge_features(x):
    x = x.copy()

    if {"reported_spd_x","reported_spd_y"}.issubset(x.columns):
        x["reported_sender_speed_magnitude"] = np.sqrt(
            x["reported_spd_x"]**2 + x["reported_spd_y"]**2
        )

    if {"reported_pos_x","reported_pos_y","receiver_pos_x","receiver_pos_y"}.issubset(x.columns):
        x["reported_distance_to_receiver"] = np.sqrt(
            (x["reported_pos_x"]-x["receiver_pos_x"])**2 +
            (x["reported_pos_y"]-x["receiver_pos_y"])**2
        )

    if {"reported_spd_x","reported_spd_y","receiver_spd_x","receiver_spd_y"}.issubset(x.columns):
        x["relative_velocity_magnitude"] = np.sqrt(
            (x["reported_spd_x"]-x["receiver_spd_x"])**2 +
            (x["reported_spd_y"]-x["receiver_spd_y"])**2
        )

    if "communication_delay" not in x.columns:
        x["communication_delay"] = _safe_num(x["rcvTime"]) - _safe_num(x["sendTime"])

    x = x.sort_values(["sender","receiver","rcvTime","messageID"]).reset_index(drop=True)
    pair = x.groupby(["sender","receiver"], sort=False)
    x["prior_pair_message_count"] = pair.cumcount()
    x["time_since_previous_pair_message"] = pair["rcvTime"].diff()

    x = x.sort_values(["sender","rcvTime","messageID"]).reset_index(drop=True)
    x["prior_sender_message_count"] = x.groupby("sender", sort=False).cumcount()

    x = x.sort_values(["receiver","rcvTime","messageID"]).reset_index(drop=True)
    x["prior_receiver_message_count"] = x.groupby("receiver", sort=False).cumcount()

    return x.sort_values(["snapshot_id","rcvTime","messageID"]).reset_index(drop=True)


In [ ]:
def topology_features_for_snapshot(snapshot_edges, active_nodes=None):
    if active_nodes is None:
        active_nodes = []

    active_nodes = {int(v) for v in active_nodes if pd.notna(v)}

    dg = nx.DiGraph()
    dg.add_nodes_from(active_nodes)

    if not snapshot_edges.empty:
        pairs = (
            snapshot_edges[["sender","receiver"]]
            .dropna()
            .drop_duplicates()
            .astype("int64")
        )
        dg.add_edges_from(pairs.itertuples(index=False, name=None))

    ug = dg.to_undirected()
    nodes = sorted(dg.nodes())
    n = len(nodes)
    unique_edges = dg.number_of_edges()

    indeg = dict(dg.in_degree())
    outdeg = dict(dg.out_degree())
    degree = dict(ug.degree())

    pagerank = nx.pagerank(dg) if n > 0 else {}
    clustering = nx.clustering(ug) if n > 0 else {}
    components = list(nx.connected_components(ug)) if n > 0 else []

    component_size = {}
    for comp in components:
        for v in comp:
            component_size[v] = len(comp)

    denom = max(n-1, 1)

    node_topology = pd.DataFrame({
        "vehicle_id": nodes,
        "in_degree": [indeg.get(v,0) for v in nodes],
        "out_degree": [outdeg.get(v,0) for v in nodes],
        "degree": [degree.get(v,0) for v in nodes],
        "degree_centrality": [degree.get(v,0)/denom for v in nodes],
        "pagerank": [pagerank.get(v,0.0) for v in nodes],
        "clustering_coefficient": [clustering.get(v,0.0) for v in nodes],
        "component_size": [component_size.get(v,1) for v in nodes],
    })

    density = float(nx.density(dg)) if n > 1 else 0.0

    if unique_edges == 0:
        reciprocity = 0.0
    else:
        reciprocity = nx.reciprocity(dg)
        reciprocity = 0.0 if reciprocity is None else float(reciprocity)

    avg_degree = float(np.mean(list(degree.values()))) if degree else 0.0
    avg_clustering = float(nx.average_clustering(ug)) if n > 0 else 0.0
    largest_component = max((len(c) for c in components), default=0)
    largest_fraction = largest_component/n if n else 0.0
    isolated_nodes = sum(1 for v in nodes if degree.get(v,0) == 0)

    mean_rssi = (
        float(pd.to_numeric(snapshot_edges["RSSI"], errors="coerce").mean())
        if (not snapshot_edges.empty and "RSSI" in snapshot_edges.columns)
        else np.nan
    )

    median_delay = (
        float(pd.to_numeric(snapshot_edges["communication_delay"], errors="coerce").median())
        if (not snapshot_edges.empty and "communication_delay" in snapshot_edges.columns)
        else np.nan
    )

    graph_stats = {
        "node_count": int(n),
        "isolated_node_count": int(isolated_nodes),
        "message_edge_count": int(len(snapshot_edges)),
        "unique_directed_edge_count": int(unique_edges),
        "density": density,
        "reciprocity": reciprocity,
        "average_degree": avg_degree,
        "average_clustering": avg_clustering,
        "connected_components": int(len(components)),
        "largest_component_size": int(largest_component),
        "largest_component_fraction": float(largest_fraction),
        "mean_RSSI": mean_rssi,
        "median_communication_delay": median_delay,
    }

    if not snapshot_edges.empty and "attackerType" in snapshot_edges.columns:
        y = pd.to_numeric(snapshot_edges["attackerType"], errors="coerce")
        graph_stats["labelled_attack_edge_fraction"] = float((y != 0).mean())
    else:
        graph_stats["labelled_attack_edge_fraction"] = np.nan

    return node_topology, graph_stats


In [ ]:
def compute_archive_topology(edges, snapshot_nodes):
    node_parts = []
    graph_rows = []

    snapshot_ids = sorted(
        set(edges["snapshot_id"].dropna().astype(int))
        |
        set(snapshot_nodes["snapshot_id"].dropna().astype(int))
    )

    for snapshot_id in snapshot_ids:
        snap_edges = edges[edges["snapshot_id"] == snapshot_id]
        snap_nodes = snapshot_nodes[snapshot_nodes["snapshot_id"] == snapshot_id]

        active_nodes = (
            snap_nodes["vehicle_id"]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        nt, gs = topology_features_for_snapshot(
            snap_edges,
            active_nodes
        )

        if not nt.empty:
            nt["snapshot_id"] = int(snapshot_id)
            node_parts.append(nt)

        gs["snapshot_id"] = int(snapshot_id)

        if not snap_edges.empty:
            gs["snapshot_start"] = float(snap_edges["snapshot_start"].iloc[0])
            gs["snapshot_end"] = float(snap_edges["snapshot_end"].iloc[0])
        else:
            gs["snapshot_start"] = np.nan
            gs["snapshot_end"] = np.nan

        graph_rows.append(gs)

    node_topology = pd.concat(node_parts, ignore_index=True) if node_parts else pd.DataFrame()
    graph_features = pd.DataFrame(graph_rows)

    return node_topology, graph_features


def add_lagged_topology(node_topology):
    if node_topology.empty:
        return node_topology.copy()

    x = node_topology.sort_values(["vehicle_id","snapshot_id"]).copy()
    g = x.groupby("vehicle_id", sort=False)

    for col in [
        "in_degree","out_degree","degree","degree_centrality",
        "pagerank","clustering_coefficient","component_size"
    ]:
        x[f"prev_{col}"] = g[col].shift(1)

    return x


def add_lagged_graph_features(graph_features):
    if graph_features.empty:
        return graph_features.copy()

    x = graph_features.sort_values("snapshot_id").copy()

    for col in [
        "node_count","isolated_node_count","message_edge_count",
        "unique_directed_edge_count","density","reciprocity",
        "average_degree","average_clustering","connected_components",
        "largest_component_size","largest_component_fraction",
        "mean_RSSI","median_communication_delay"
    ]:
        if col in x.columns:
            x[f"prev_{col}"] = x[col].shift(1)

    return x


def merge_node_state_and_topology(snapshot_nodes, topology):
    if snapshot_nodes.empty:
        return topology.copy()
    if topology.empty:
        return snapshot_nodes.copy()

    return snapshot_nodes.merge(
        topology,
        on=["snapshot_id","vehicle_id"],
        how="outer",
        validate="one_to_one"
    )


## Three-archive validation

Run this before the full production cell. It should report:
- `Completed: 3`
- `Failed: 0`
- `snapshots = 100`
- `node_topology_nulls = 0`
- `negative_delays = 0`


In [ ]:
test_summaries = []
test_failures = []

for i, archive_dir in enumerate(archive_dirs[:3], start=1):
    print(f"[{i}/3] {archive_dir.name}")

    try:
        nodes = pd.read_parquet(archive_dir / "node_states.parquet")
        edges = pd.read_parquet(archive_dir / "communication_edges.parquet")

        nodes_with_snapshot = add_snapshot_to_nodes(nodes, edges, WINDOW_SIZE)

        # FINAL RULE: exclude initialization-only pre-communication snapshot -1.
        nodes_with_snapshot = nodes_with_snapshot[
            nodes_with_snapshot["snapshot_id"] >= 0
        ].copy()

        snapshot_nodes = latest_node_state_per_snapshot(nodes_with_snapshot)

        edges_with_receiver = add_receiver_state_to_edges(edges, nodes)
        edge_features = engineer_edge_features(edges_with_receiver)

        node_topology, graph_features = compute_archive_topology(
            edge_features,
            snapshot_nodes
        )

        node_topology = add_lagged_topology(node_topology)
        graph_features = add_lagged_graph_features(graph_features)

        node_features = merge_node_state_and_topology(
            snapshot_nodes,
            node_topology
        )

        topology_cols = [
            "in_degree","out_degree","degree","degree_centrality",
            "pagerank","clustering_coefficient","component_size"
        ]

        topology_nulls = int(
            node_features[topology_cols].isna().sum().sum()
        )

        negative_delays = int(
            (edge_features["communication_delay"] < 0).sum()
        )

        test_summaries.append({
            "archive_name": archive_dir.name,
            "node_features": len(node_features),
            "edge_features": len(edge_features),
            "snapshots": len(graph_features),
            "node_topology_nulls": topology_nulls,
            "negative_delays": negative_delays,
        })

        print("SUCCESS")

    except Exception as e:
        test_failures.append({
            "archive_name": archive_dir.name,
            "error": repr(e)
        })
        print("FAILED:", repr(e))

test_summary_df = pd.DataFrame(test_summaries)
test_failure_df = pd.DataFrame(test_failures)

print("\nCompleted:", len(test_summary_df))
print("Failed:", len(test_failure_df))
display(test_summary_df)
display(test_failure_df)


## Full production run

Run only after the 3-archive validation succeeds.  
This cell saves the final corrected Parquet files to:

`GRAPH_FEATURE_ROOT / "archives" / ...`


In [ ]:
summaries = []
failures = []

for i, archive_dir in enumerate(archive_dirs, start=1):
    print(f"[{i}/{len(archive_dirs)}] {archive_dir.name}")

    try:
        nodes = pd.read_parquet(archive_dir / "node_states.parquet")
        edges = pd.read_parquet(archive_dir / "communication_edges.parquet")

        nodes_with_snapshot = add_snapshot_to_nodes(nodes, edges, WINDOW_SIZE)

        # FINAL TEMPORAL DEFINITION: retain only snapshots 0..99.
        nodes_with_snapshot = nodes_with_snapshot[
            nodes_with_snapshot["snapshot_id"] >= 0
        ].copy()

        snapshot_nodes = latest_node_state_per_snapshot(nodes_with_snapshot)

        edges_with_receiver = add_receiver_state_to_edges(edges, nodes)
        edge_features = engineer_edge_features(edges_with_receiver)

        node_topology, graph_features = compute_archive_topology(
            edge_features,
            snapshot_nodes
        )

        node_topology = add_lagged_topology(node_topology)
        graph_features = add_lagged_graph_features(graph_features)

        node_features = merge_node_state_and_topology(
            snapshot_nodes,
            node_topology
        )

        topology_cols = [
            "in_degree","out_degree","degree","degree_centrality",
            "pagerank","clustering_coefficient","component_size"
        ]

        topology_nulls = int(
            node_features[topology_cols].isna().sum().sum()
        )

        negative_delays = int(
            (edge_features["communication_delay"] < 0).sum()
        )

        if topology_nulls != 0:
            raise ValueError(f"Current topology nulls: {topology_nulls}")

        if negative_delays != 0:
            raise ValueError(f"Negative delays: {negative_delays}")

        if len(graph_features) != 100:
            raise ValueError(f"Unexpected snapshot count: {len(graph_features)}")

        out_dir = FEATURE_ROOT / "archives" / archive_dir.name
        out_dir.mkdir(parents=True, exist_ok=True)

        node_features.to_parquet(
            out_dir / "node_features.parquet",
            index=False,
            compression="zstd"
        )

        edge_features.to_parquet(
            out_dir / "edge_features.parquet",
            index=False,
            compression="zstd"
        )

        graph_features.to_parquet(
            out_dir / "snapshot_features.parquet",
            index=False,
            compression="zstd"
        )

        summaries.append({
            "archive_name": archive_dir.name,
            "node_feature_rows": len(node_features),
            "edge_feature_rows": len(edge_features),
            "snapshot_rows": len(graph_features),
            "receiver_state_coverage": float(
                edge_features["receiver_state_age"].notna().mean()
            ) if "receiver_state_age" in edge_features.columns else np.nan,
            "current_topology_nulls": topology_nulls,
            "negative_delay_count": negative_delays,
            "node_null_cells": int(node_features.isna().sum().sum()),
            "edge_null_cells": int(edge_features.isna().sum().sum()),
            "snapshot_null_cells": int(graph_features.isna().sum().sum()),
        })

        print("SUCCESS")

    except Exception as e:
        failures.append({
            "archive_name": archive_dir.name,
            "error": repr(e)
        })
        print("FAILED:", repr(e))

summary_df = pd.DataFrame(summaries)
failure_df = pd.DataFrame(failures)

summary_df.to_csv(
    FEATURE_ROOT / "feature_engineering_summary.csv",
    index=False
)

failure_df.to_csv(
    FEATURE_ROOT / "feature_engineering_failures.csv",
    index=False
)

print("\n" + "="*70)
print("FULL PRODUCTION RUN COMPLETE")
print("="*70)
print("Completed:", len(summary_df))
print("Failed:", len(failure_df))
display(summary_df.head())
display(failure_df.head())


In [ ]:
print("="*70)
print("FINAL GLOBAL VALIDATION")
print("="*70)

print("Archives completed:", len(summary_df))
print("Archives failed:", len(failure_df))

print(
    "Total node-feature rows:",
    int(summary_df["node_feature_rows"].sum())
)

print(
    "Total edge-feature rows:",
    int(summary_df["edge_feature_rows"].sum())
)

print(
    "Total graph snapshots:",
    int(summary_df["snapshot_rows"].sum())
)

print(
    "Total current-topology nulls:",
    int(summary_df["current_topology_nulls"].sum())
)

print(
    "Total negative delays:",
    int(summary_df["negative_delay_count"].sum())
)

print(
    "Minimum receiver-state coverage:",
    summary_df["receiver_state_coverage"].min()
)


In [ ]:
con = duckdb.connect()

edge_glob = (
    FEATURE_ROOT / "archives" / "*" / "edge_features.parquet"
).as_posix()

node_glob = (
    FEATURE_ROOT / "archives" / "*" / "node_features.parquet"
).as_posix()

snapshot_glob = (
    FEATURE_ROOT / "archives" / "*" / "snapshot_features.parquet"
).as_posix()

global_counts = con.execute(f"""
SELECT
    (SELECT COUNT(*) FROM read_parquet('{edge_glob}', union_by_name=true)) AS edge_rows,
    (SELECT COUNT(*) FROM read_parquet('{node_glob}', union_by_name=true)) AS node_rows,
    (SELECT COUNT(*) FROM read_parquet('{snapshot_glob}', union_by_name=true)) AS snapshot_rows
""").df()

display(global_counts)

global_counts.to_csv(
    FEATURE_ROOT / "global_feature_counts.csv",
    index=False
)


In [ ]:
snapshot_characterization = con.execute(f"""
SELECT
    COUNT(*) AS snapshot_count,
    AVG(node_count) AS mean_nodes,
    MEDIAN(node_count) AS median_nodes,
    AVG(isolated_node_count) AS mean_isolated_nodes,
    AVG(
        CASE
            WHEN node_count > 0
            THEN isolated_node_count::DOUBLE / node_count
            ELSE 0
        END
    ) AS mean_isolated_node_fraction,
    AVG(message_edge_count) AS mean_message_edges,
    AVG(unique_directed_edge_count) AS mean_unique_edges,
    AVG(density) AS mean_density,
    AVG(reciprocity) AS mean_reciprocity,
    AVG(average_degree) AS mean_degree,
    AVG(average_clustering) AS mean_clustering,
    AVG(connected_components) AS mean_components,
    AVG(largest_component_fraction) AS mean_largest_component_fraction
FROM read_parquet('{snapshot_glob}', union_by_name=true)
""").df()

display(snapshot_characterization)

snapshot_characterization.to_csv(
    FEATURE_ROOT / "global_snapshot_characterization.csv",
    index=False
)


In [ ]:
edge_characterization = con.execute(f"""
SELECT
    COUNT(*) AS edge_rows,
    AVG(communication_delay) AS mean_delay,
    MEDIAN(communication_delay) AS median_delay,
    QUANTILE_CONT(communication_delay, 0.95) AS p95_delay,
    AVG(RSSI) AS mean_RSSI,
    MEDIAN(RSSI) AS median_RSSI,
    AVG(reported_distance_to_receiver) AS mean_reported_distance,
    MEDIAN(reported_distance_to_receiver) AS median_reported_distance,
    AVG(relative_velocity_magnitude) AS mean_relative_velocity
FROM read_parquet('{edge_glob}', union_by_name=true)
""").df()

display(edge_characterization)

edge_characterization.to_csv(
    FEATURE_ROOT / "global_edge_characterization.csv",
    index=False
)
